# P3 Movie Genre — Notebook 1: Exploratory Data Analysis

**Goal:** understand the movies, the genre labels, and the plot text in `mpst_imdb_movies.csv` before any modeling.

This notebook only explores — no vectorizing, no modeling. Its one hand-off to Notebooks 2 and 3 is the **vote filter** (`VOTE_MIN`), chosen in §1.3.

| Section | Contents |
|---|---|
| 1 | Movie characteristics: metadata distributions, rating by genre, vote filter |
| 2 | Genre / class balance: positive counts and rates, genre correlation |
| 3 | Text characteristics: synopsis length, distinguishing vocabulary |

`startYear`, `runtimeMinutes`, `averageRating`, and `numVotes` are used here for analysis and filtering only. They are never model inputs.

In [ ]:
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)

DATA_PATH = Path("../data/mpst_imdb_movies.csv")

**Data location.** The CSV lives in `data/` at the repo root, which `.gitignore` excludes, so the provided dataset is never committed. Download it from the class Google Drive (`Datasets` folder) into `data/` before running.

In [ ]:
df = pd.read_csv(DATA_PATH)

META_COLS = ["imdb_id", "primaryTitle", "plot_synopsis", "synopsis_source", "split",
             "startYear", "runtimeMinutes", "averageRating", "numVotes"]
NUM_COLS = ["startYear", "runtimeMinutes", "averageRating", "numVotes"]
GENRE_COLS = [c for c in df.columns if c not in META_COLS]

df[NUM_COLS] = df[NUM_COLS].apply(pd.to_numeric, errors="coerce")

print(f"{len(df):,} movies, {len(GENRE_COLS)} genre columns")
print("Genres:", ", ".join(GENRE_COLS))

Every column that isn't a known metadata column is a genre dummy, so `GENRE_COLS` is derived rather than typed by hand. The numeric metadata is coerced to numbers because IMDB encodes missing values as `\N`, which would otherwise load as text.

In [ ]:
pd.concat([df[META_COLS].isna().sum().rename("missing"),
           df[META_COLS].nunique().rename("unique")], axis=1).T

In [ ]:
df["split"].value_counts().to_frame("movies")

## 1. Movie characteristics

### 1.1 Metadata distributions
`numVotes` spans several orders of magnitude, so it's plotted on a log scale. Runtime is clipped at 240 minutes for display only, so a handful of extreme values don't squash the histogram.

In [ ]:
df[NUM_COLS].describe().T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df["startYear"], bins=40, ax=axes[0, 0])
sns.histplot(df["runtimeMinutes"].clip(upper=240), bins=40, ax=axes[0, 1])
sns.histplot(df["averageRating"], bins=40, ax=axes[1, 0])
sns.histplot(df["numVotes"], bins=50, log_scale=True, ax=axes[1, 1])
axes[0, 0].set_title("Release year")
axes[0, 1].set_title("Runtime (minutes, clipped at 240)")
axes[1, 0].set_title("Average IMDB rating")
axes[1, 1].set_title("Number of votes (log scale)")
plt.tight_layout()
plt.show()

**TODO (write after running):** describe each distribution — e.g. which decades dominate the corpus, the typical runtime, whether ratings are centered/skewed, and the range of `numVotes`.

### 1.2 Average rating by genre
A movie with several genres contributes to several boxes. Genres are ordered by median rating.

In [ ]:
rating_long = (df.melt(id_vars="averageRating", value_vars=GENRE_COLS,
                       var_name="genre", value_name="flag")
                 .query("flag == 1"))
order = rating_long.groupby("genre")["averageRating"].median().sort_values(ascending=False).index

plt.figure(figsize=(14, 5))
sns.boxplot(data=rating_long, x="genre", y="averageRating", order=order, fliersize=2)
plt.xticks(rotation=60, ha="right")
plt.title("Average IMDB rating by genre")
plt.tight_layout()
plt.show()

**TODO (write after running):** which genres rate highest and lowest, and how much the boxes overlap. (Typically Horror rates lowest and Documentary/Biography/History highest.)

### 1.3 Vote filter
Movies with very few votes tend to have unstable ratings and less reliable genre tags. To choose a minimum, first look at how many rows each candidate threshold would remove from each split.

In [ ]:
candidates = [100, 500, 1000, 2500, 5000]
pd.DataFrame({f"< {t:,}": (df["numVotes"] < t).groupby(df["split"]).sum() for t in candidates})

In [ ]:
VOTE_MIN = 1000

below = (df["numVotes"] < VOTE_MIN).groupby(df["split"])
drop_report = pd.DataFrame({"dropped": below.sum(), "total": below.size()})
drop_report["pct_dropped"] = 100 * drop_report["dropped"] / drop_report["total"]
drop_report.loc["all"] = [drop_report["dropped"].sum(), drop_report["total"].sum(),
                          100 * drop_report["dropped"].sum() / drop_report["total"].sum()]
drop_report.round(1)

**Decision: `VOTE_MIN = 1000`.** At around a thousand votes, a film's rating and genre tags have been confirmed by a reasonable number of users. The threshold is deliberately modest so the corpus stays large for training. The table above reports the rows dropped per split; the drop should be a small share of each split and roughly proportional across splits, so the train/test balance is unchanged.

**TODO (confirm after running):** state the actual percentage dropped. If it's more than about 5–10%, lower `VOTE_MIN` to 500 and update the same constant in Notebooks 2 and 3.

Notebooks 2 and 3 apply this same filter. The rest of this notebook uses the filtered data, so what we see here matches what the models see.

In [ ]:
df = df[df["numVotes"] >= VOTE_MIN].reset_index(drop=True)
print(f"{len(df):,} movies after the vote filter")

## 2. Genre / class balance

### 2.1 Positive counts and rates for all genres
The red dashed line marks 5%, the minimum positive rate Notebook 2 requires for a selected genre.

In [ ]:
counts = df[GENRE_COLS].sum().sort_values(ascending=False)
rates = 100 * counts / len(df)

fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
sns.barplot(x=counts.values, y=counts.index, ax=axes[0], color="steelblue")
sns.barplot(x=rates.values, y=rates.index, ax=axes[1], color="darkorange")
axes[1].axvline(5, color="red", ls="--", label="5% threshold")
axes[0].set(title="Positive count per genre", xlabel="movies")
axes[1].set(title="Positive rate per genre", xlabel="% of movies")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
df[GENRE_COLS].sum(axis=1).value_counts().sort_index().rename_axis("genres per movie").to_frame("movies").T

**TODO (write after running):** which genres are common (Drama is usually dominant) and which fall below 5%. Note the typical number of genres per movie: since IMDB allows up to three, each genre is its own imbalanced yes/no problem, which is why Notebook 2 trains one binary classifier per genre.

### 2.2 Genre correlation
Pearson correlation between the 0/1 dummies (the phi coefficient). Only the lower triangle is shown, since the matrix is symmetric.

In [ ]:
corr = df[GENRE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 10))
sns.heatmap(corr, mask=mask, cmap="vlag", center=0, vmin=-0.5, vmax=0.5,
            square=True, linewidths=0.3, cbar_kws={"shrink": 0.7})
plt.title("Correlation between genre dummies")
plt.tight_layout()
plt.show()

**TODO (write after running):** call out the strongest positive pairs (e.g. Crime–Thriller, Action–Adventure, Animation–Family) and the strongest negative ones (e.g. Comedy vs. Horror/Thriller). Strongly correlated genres share vocabulary, so a classifier for one will partly fire on the other.

## 3. Text characteristics and vocabulary

### 3.1 Synopsis length
Approximate tokens use the common rule of thumb of **≈ 4 characters per token** for English sub-word tokenizers. Synopsis length matters downstream, because sentence-embedding models and LLMs have fixed context windows.

In [ ]:
chars = df["plot_synopsis"].str.len()
lengths = pd.DataFrame({
    "characters": chars,
    "words": df["plot_synopsis"].str.split().str.len(),
    "approx_tokens": (chars / 4).round(),
})
lengths.agg(["mean", "median", "min", "max"]).T.round(0)

In [ ]:
plt.figure(figsize=(10, 4))
sns.histplot(x=lengths["approx_tokens"], hue=df["synopsis_source"], bins=60, log_scale=True, element="step")
plt.title("Synopsis length by source (approx. tokens, log scale)")
plt.xlabel("approx. tokens")
plt.tight_layout()
plt.show()

**TODO (write after running):** report the typical and extreme lengths, and whether Wikipedia and IMDB synopses differ. A long right tail means some synopses will need truncation or chunking for fixed-context models.

### 3.2 Vocabulary: top terms within genres vs. overall
Each synopsis is tokenized into its set of distinct lowercase words (3+ letters, English stop words removed). **Document frequency** is the share of synopses containing a word. This is plain counting for exploration, not a feature matrix.

Raw frequency alone tends to surface the same narrative words in every genre ("finds", "tells", "kills"). So we also compute **lift** = (document frequency within the genre) / (document frequency overall), restricted to words that appear in at least 5% of the genre's synopses. Lift above 1 means a word is over-represented in that genre.

In [ ]:
TOKEN_RE = re.compile(r"[a-z]{3,}")
doc_sets = df["plot_synopsis"].str.lower().map(lambda t: set(TOKEN_RE.findall(t)) - ENGLISH_STOP_WORDS)


def doc_freq(mask):
    counter = Counter()
    for words in doc_sets[mask]:
        counter.update(words)
    return pd.Series(counter, dtype=float) / mask.sum()


overall_df = doc_freq(pd.Series(True, index=df.index))
SHOWCASE = [g for g in ["Horror", "Comedy", "Romance", "Sci-Fi"] if g in GENRE_COLS]
genre_df = {g: doc_freq(df[g] == 1) for g in SHOWCASE}

top_raw = pd.DataFrame({"Overall": overall_df.nlargest(10).index}
                       | {g: genre_df[g].nlargest(10).index for g in SHOWCASE})
top_raw.index = range(1, 11)
top_raw

The ten most common words in each genre (above) overlap heavily with the overall list. That's why raw frequency is not enough, and why the lift plots below are needed.

In [ ]:
fig, axes = plt.subplots(1, len(SHOWCASE), figsize=(4.2 * len(SHOWCASE), 5))
for ax, g in zip(np.atleast_1d(axes), SHOWCASE):
    supported = genre_df[g][genre_df[g] >= 0.05]
    lift = (supported / overall_df[supported.index]).nlargest(12)
    sns.barplot(x=lift.values, y=lift.index, ax=ax, color="seagreen")
    ax.axvline(1, color="grey", ls="--")
    ax.set(title=f"{g}: highest lift", xlabel="lift vs. overall")
plt.tight_layout()
plt.show()

**TODO (write after running):** which words separate each genre (e.g. Horror: *demon, ghost, possessed*; Romance: *marry, wedding, love*; Sci-Fi: *planet, alien, earth*). Also note any words that are **character or franchise names** with high lift. They separate classes in-sample but won't generalize, and the TF-IDF model in Notebook 2 will learn them too.

## Summary

**TODO (write after running):** 4–6 bullets covering: corpus size after the vote filter, the dominant genres and the imbalance, the main genre correlations, synopsis length and its implications, and whether distinguishing vocabulary exists (which justifies a bag-of-words model in Notebook 2).

**Hand-off to Notebooks 2 and 3:** `VOTE_MIN = 1000`.